# Pandas for Machine Learning & Data Analysis

<!-- This notebook teaches Pandas the way you'll actually use it in an ML workflow — organized by **what you do to data**, in the order you'd typically do it, not by abstract topic.

## Roadmap

| Part | Skill | Why it matters for ML |
|---|---|---|
| 1 | Series & DataFrame Fundamentals | The two structures everything else builds on |
| 2 | Loading & First Contact with Data | You can't do anything before the data is in and understood |
| 3 | Selection and Filtering | The single most-used Pandas skill in ML — slicing, subsetting, debugging inputs |
| 4 | Missing Data Handling | Most scikit-learn models crash on `NaN` — this is non-negotiable |
| 5 | Data Types and Conversion | A numeric column stored as text will silently break your model |
| 6 | Value Exploration | Is your target balanced? Is a category too high-cardinality to encode? |
| 7 | Handling Outliers | Extreme values distort means, correlations, and distance-based models |
| 8 | Cleaning Up — Rename, Index, Drop | Housekeeping that prevents downstream bugs |
| 9 | Feature Engineering | Creating the columns your model will actually learn from |
| 10 | Encoding Categoricals | The bridge between Pandas and a model that only understands numbers |
| 11 | GroupBy and Aggregation | How you discover patterns before modeling |
| 12 | Sorting and Ranking | Finding top/bottom records for EDA |
| 13 | Duplicates | Duplicate rows inflate training data and leak information |
| 14 | Merging, Concatenation & Reshaping | Real ML data almost never lives in one table |
| 15 | Correlation Analysis | Spot multicollinearity and target-relevant features |
| 16 | String Operations | Dirty text has to be cleaned before it can be encoded |
| 17 | Index Management | The #1 source of confusing `KeyError`s for beginners |
| 18 | Scaling and Normalization | Puts features on comparable scales for distance- and gradient-based models |
| 19 | Sampling and Splitting | The literal last Pandas step before scikit-learn |


Every concept is **taught** using the real **Titanic passenger dataset** — one dataset, start to finish, so you can focus entirely on the technique. Then every Part ends with a short **task** asking you to apply that same skill to a different real dataset (**Palmer Penguins**), with a worked **answer** included, so you can check that the skill actually transferred rather than just following along. A **Capstone Project** closes out the notebook so you can prove it yourself, end to end. -->

### What is Pandas?

Think of Pandas as **"Excel inside Python"**:
- A Pandas **DataFrame** is like a spreadsheet — rows and columns, with headers
- A Pandas **Series** is like a single column in that spreadsheet
- Just like Excel, you can filter, sort, sum, and summarize your data — except now it's done with code, so it's repeatable, automatable, and scales to millions of rows.

Pandas is an open-source Python library used for **data manipulation and analysis**, built on top of **NumPy**.

### Why do we use Pandas?

- Handles real-world, messy data (CSV, Excel, JSON, SQL databases, etc.)
- Fast, vectorized operations (built on NumPy — much faster than pure Python loops)
- Powerful tools for cleaning, filtering, grouping, and merging data
- Much less code than plain Python for the same tasks
- The standard first step in almost every Data Analysis / Machine Learning pipeline

In [ ]:
# Uncomment the line below if pandas is not installed
# !pip install pandas

import pandas as pd
print("Pandas version:", pd.__version__)

### NumPy vs. Pandas — What's the Difference?

Pandas is actually **built on top of NumPy**, but they solve different problems.

| | NumPy | Pandas |
|---|---|---|
| Core structure | `ndarray` (n-dimensional array) | `Series` (1D) / `DataFrame` (2D) |
| Data types | One dtype for the whole array | A different dtype per column |
| Labels | None — only integer positions | Row index + column names |
| Missing data | No built-in handling | Built-in: `.isnull()`, `.fillna()`, `.dropna()` |
| Best for | Fast numerical / matrix computation | Labeled, real-world, mixed-type tabular data |

**Use NumPy when:**
- You're doing heavy math: linear algebra, matrix multiplication, statistics on raw numbers
- Your data is homogeneous (all one type) — e.g. images, audio signals, sensor readings
- You need maximum speed and minimum memory, and don't need row/column labels
- You're feeding numeric arrays directly into a model (scikit-learn, TensorFlow, and PyTorch all use NumPy arrays under the hood)

**Use Pandas when:**
- You're loading and exploring a real-world dataset (CSV, Excel, SQL, JSON)
- Your columns have different types (numbers, text, dates, categories) — like the datasets you'll load in a moment
- You need to clean, filter, group, merge, or reshape data
- You want readable output with meaningful row/column labels instead of raw numbers

In practice: **Pandas for loading, cleaning, and exploring data. NumPy for the math underneath** (and Pandas uses it internally for every column).

In [ ]:
import numpy as np

sample_array = np.array([10, 20, 30, 40])                # a NumPy array: just numbers, no labels
sample_series = pd.Series(sample_array, name="values")   # a Pandas Series: numbers + index + name

print(type(sample_array), "-> Mean (NumPy):", np.mean(sample_array))
print(type(sample_series), "-> Mean (Pandas):", sample_series.mean())

# They're closely related: a Series is just a NumPy array with an index attached
print()
print("Same underlying values?", np.array_equal(sample_array, sample_series.to_numpy()))

### Without Pandas vs. With Pandas

**Task:** Given student scores, calculate the average score and find the student with the highest score.

In [ ]:
# ---- Without Pandas (plain Python) ----
students = [
    {"name": "Ali", "score": 85},
    {"name": "Sara", "score": 92},
    {"name": "Omar", "score": 78},
    {"name": "Laila", "score": 95},
]

total = 0
highest = students[0]
for student in students:
    total += student["score"]
    if student["score"] > highest["score"]:
        highest = student

average = total / len(students)
print("Average score:", average)
print("Highest score:", highest["name"], highest["score"])

In [ ]:
# ---- With Pandas ----
df_students = pd.DataFrame(students)

average = df_students["score"].mean()
highest = df_students.loc[df_students["score"].idxmax()]

print("Average score:", average)
print("Highest score:", highest["name"], highest["score"])

**Takeaway:** Same result, far less code with Pandas — and it scales to millions of rows without changing the approach.

## Part 1: Series & DataFrame Fundamentals

The two building blocks everything else in this notebook is built on.

### Series, DataFrame & Index

- **Series** — a one-dimensional labeled array (like a single column)
- **DataFrame** — a two-dimensional labeled table, made up of multiple Series (columns)
- **Index** — the labels attached to rows (and columns) that make lookups fast and meaningful

In [ ]:
# A Series: one column of data with an index
scores = pd.Series([85, 92, 78, 95], name="score")
print(scores)
print(type(scores))

#### More on Series: `.array`, `.index`, and Custom Labels

A Series stores its data in two parts: the **values** (accessible via `.array`) and the **labels** (accessible via `.index`).

In [ ]:
obj = pd.Series([4, 7, -5, 3])
obj

In [ ]:
obj.array

In [ ]:
obj.index

By default, a Series gets an integer index `0, 1, 2, ...`. You can assign your own labels instead of the default integers:

In [ ]:
obj2 = pd.Series([4, 7, -5, 3], index=["d", "b", "a", "c"])
obj2

In [ ]:
obj2.index

You can then use these labels to select, assign, or look up values — just like a dictionary:

In [ ]:
obj2["a"]        # access a single value by label

In [ ]:
obj2["d"] = 6    # reassign a value by label
obj2

In [ ]:
obj2[["c", "a", "d"]]   # select several labels at once

### NumPy-like Operations Preserve the Index-Value Link

Filtering with a boolean array, scalar multiplication, or applying a math function all keep each value attached to its original label:

In [ ]:
obj2[obj2 > 0]         # boolean filtering

In [ ]:
obj2 * 2               # scalar multiplication

#### Creating a Series from a Dictionary

A Python dictionary maps naturally to a Series: keys become the index, values become the data.

In [ ]:
sdata = {"Ohio": 35000, "Texas": 71000, "Oregon": 16000, "Utah": 5000}
obj3 = pd.Series(sdata)
obj3

A Series can be converted back to a dictionary with `.to_dict()`:

In [ ]:
obj3.to_dict()

When you pass only a dictionary, the index order follows the dictionary's key insertion order. You can override this by passing your own `index`:

In [ ]:
states = ["California", "Ohio", "Oregon", "Texas"]
sdata = {"Ohio": 35000, "Texas": 71000, "Oregon": 16000, "Utah": 5000}

obj4 = pd.Series(sdata, index=states)
obj4

Here, three values from `sdata` landed in the right spots, but `"California"` wasn't in `sdata`, so it shows up as `NaN` (Not a Number) — pandas' marker for missing data. `"Utah"` wasn't included in `states`, so it's excluded from the result entirely.

#### Detecting Missing Data — `pd.isna()` / `pd.notna()`

In [ ]:
pd.isna(obj4)

In [ ]:
pd.notna(obj4)

In [ ]:
obj4.isna()   # same check, called as a method on the Series

#### Automatic Alignment by Index Label

When you do arithmetic between two Series, pandas automatically aligns values by index label — similar to a database join. Labels that don't appear in **both** Series come out as `NaN`.

In [ ]:
obj3

In [ ]:
obj4

In [ ]:
obj3 + obj4   # aligned by label: Ohio/Texas/Oregon add up, California/Utah become NaN

#### Naming a Series and its Index

Both the Series itself and its index have a `.name` attribute, which shows up in the output and integrates with the rest of pandas.

In [ ]:
obj4.name = "population"
obj4.index.name = "state"
obj4

In [ ]:
# A DataFrame: a table made of multiple Series (columns)
df = pd.DataFrame({
    "name": ["Ali", "Sara", "Omar", "Laila"],
    "score": [85, 92, 78, 95],
})
print(df)
print(type(df))

In [ ]:
# Every DataFrame/Series has an Index
print(df.index)
print(df.columns)

### Creating DataFrames from Dictionaries and Lists

In [ ]:
# From a dictionary of lists (each key becomes a column)
data = {
    "name": ["Ali", "Sara", "Omar", "Laila"],
    "age": [22, 24, 21, 23],
    "score": [85, 92, 78, 95],
}
df = pd.DataFrame(data)
df

In [ ]:
# From a list of dictionaries (each dict becomes a row)
rows = [
    {"name": "Ali", "age": 22, "score": 85},
    {"name": "Sara", "age": 24, "score": 92},
]
pd.DataFrame(rows)

### First Look at the Data

## Section A — Reading CSV files (the workhorse format)

CSV (comma-separated values) is the single most common format you'll load in practice. Let's start with a clean, well-behaved file: `customers.csv`.


In [ ]:
customers = pd.read_csv("customers.csv")
customers.head() # first 5 rows


In [ ]:
customers.tail(3)     # last 3 rows

In [ ]:
customers.sample()

In [ ]:
customers.shape       # (rows, columns)

In [ ]:
customers.columns

In [ ]:
customers.info()      # column names, types, non-null counts

In [ ]:
customers.describe()  # summary statistics for numeric columns

In [ ]:
customers.columns, customers.dtypes

###  Task — Apply Part 2 (Loading & First Contact) to a New Dataset

Run the four-command habit — `.head()`, `.info()`, `.describe()`, `.shape` — on `dirty_students`.

In [ ]:
dirty_students = pd.read_csv("dirty_students.csv")


In [ ]:
dirty_students.head()


In [ ]:
dirty_students.info()


In [ ]:



dirty_students.describe()


In [ ]:
print(dirty_students.shape)

### Real data is messy: wrong delimiters & hidden missing values

`orders_messy.csv` looks like a CSV, but real-world exports (especially from European locales/Excel) often use **semicolons** instead of commas, and use placeholder text like `"?"`, `"N/A"`, or `"none"` for missing values instead of leaving cells blank.


In [ ]:
messy_naive = pd.read_csv("orders_messy.csv")
messy_naive

In [ ]:
# Step 1: tell pandas the real delimiter
messy = pd.read_csv("orders_messy.csv", sep=";")
messy


# ***Selection & Filtering with pandas***

DataFrame.loc[]

In [ ]:
from sklearn import datasets
import pandas as pd

# Load the iris dataset
iris = datasets.load_iris()

# Convert to a pandas DataFrame
# The data is in iris.data and feature names in iris.feature_names
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Add the target variable (species) as a column
# iris.target contains numerical labels, iris.target_names provides the actual names
iris_df['species'] = iris.target
iris_df['species'] = iris_df['species'].apply(lambda x: iris.target_names[x])

# Display the first few rows of the DataFrame
print(iris_df.head())

If you would like to read only certain columns

In [ ]:
new_df = iris_df[["sepal length (cm)", "sepal width (cm)"]]
print(new_df.head())

In [ ]:
iris_df[10:20]

In [ ]:
# Select a single column (Series)
iris_df['sepal length (cm)'].head()

In [ ]:
# df.loc[...] — label-based selection (rows/columns by name)
print("Rows 0 to 2 and columns 'sepal length (cm)' to 'petal length (cm)' using .loc:")
iris_df.loc[0:2, 'sepal length (cm)':'petal length (cm)']

In [ ]:
# df.iloc[...] — integer-position-based selection (rows/columns by index number)
print("Rows 0 to 2 and columns 0 to 2 using .iloc:")
iris_df.iloc[0:3, 0:3] # Note: iloc is exclusive on the upper bound


# --- Boolean / Conditional Filtering ---


In [ ]:


print(iris_df[iris_df['sepal length (cm)'] > 7.0].head())


In [ ]:
# df.query('col > x') — filter using a string expression
print("Rows where 'petal width (cm)' < 0.3 using .query:")
iris_df.query('`petal width (cm)` < 0.3').head()


In [ ]:
# df['col'].isin([...]) — filter rows where a column matches a list of values
print("Rows where 'species' is 'setosa' or 'versicolor' using .isin:")
iris_df[iris_df['species'].isin(['setosa', 'versicolor'])]


In [ ]:
# df['col'].str.contains('text') — filter rows where a string column matches a pattern
print("Rows where 'species' contains 'set' using .str.contains:")
iris_df[iris_df['species'].str.contains('set')]


In [ ]:
# &, | — combine multiple conditions (AND / OR)
print("Rows where 'sepal length (cm)' > 5.0 AND 'species' == 'setosa':")
iris_df[(iris_df['sepal length (cm)'] > 5.0) & (iris_df['species'] == 'setosa')].head()


*Missing-value based filtering *

In [ ]:
print("Rows with missing values in 'sepal length (cm)' (expected empty):")
iris_df[iris_df['sepal length (cm)'].isnull()]


In [ ]:

print("Rows with non-missing values in 'sepal length (cm)' (expected full):")
iris_df[iris_df['sepal length (cm)'].notnull()]


In [ ]:
print("Dropping 'species' column (first 5 rows shown):")
iris_df.drop(columns=['species'])

In [ ]:
# df.drop(columns=[...]) / df.drop(index=[...]) — remove rows/columns

print("Dropping rows with index 0 and 1 (first 5 rows shown):")
iris_df.drop(index=[0, 1])


 Setting an Index Column (index_col)

In [ ]:
df = pd.read_csv("people.csv", index_col="First Name")

 Handling Missing Values Using read_csv

In [ ]:
df = pd.read_csv("people.csv", na_values=["N/A", "Unknown"])


Using nrows in read_csv()

df = pd.read_csv('people.csv', nrows=3)
df

**skip rows**

In [ ]:
df= pd.read_csv("people.csv")
print("Previous Dataset: ")
print(df)

df = pd.read_csv("people.csv", skiprows = [4,5])
print("Dataset After skipping rows: ")
print(df)

In [ ]:
import pandas as pd

# --- Create Sample DataFrames ---
df1 = pd.DataFrame({
    'key': ['A', 'B', 'C', 'D'],
    'value1': [1, 2, 3, 4]
})

df2 = pd.DataFrame({
    'key': ['B', 'D', 'E', 'F'],
    'value2': [5, 6, 7, 8]
})

In [ ]:
df1

In [ ]:
df2

In [ ]:
# --- 1. Inner Join ---
# An inner join returns only the rows where there is a match in *both* DataFrames
# based on the specified key(s).
print("### Inner Join ###")
inner_join = pd.merge(df1, df2, on='key', how='inner')
inner_join

In [ ]:
# --- 2. Left Join ---
# A left join returns all rows from the *left* DataFrame (df1), and the matching rows
# from the *right* DataFrame (df2). If there's no match in df2, NaN (Not a Number)
# is used for the columns from df2.
# print("### Left Join ###")
left_join = pd.merge(df1, df2, on='key', how='left')
# print("Result of Left Join:")
left_join
# print("Explanation: All rows from df1 are kept. For 'A' and 'C', there's no match in df2, so 'value2' is NaN.")


In [ ]:


# --- 3. Right Join ---
# A right join returns all rows from the *right* DataFrame (df2), and the matching rows
# from the *left* DataFrame (df1). If there's no match in df1, NaN is used for
# the columns from df1.
# print("### Right Join ###")
right_join = pd.merge(df1, df2, on='key', how='right')
# print("Result of Right Join:")
right_join
# print("Explanation: All rows from df2 are kept. For 'E' and 'F', there's no match in df1, so 'value1' is NaN.")
# print("\n" + "-"*30 + "\n")

In [ ]:






# --- 4. Outer Join ---
# An outer join returns all rows when there is a match in *either* the left or
# right DataFrame. If a key is present in one DataFrame but not the other,
# the unmatched columns will have NaN.
# print("### Outer Join ###")
outer_join = pd.merge(df1, df2, on='key', how='outer')
# print("Result of Outer Join:")
outer_join
# print("Explanation: All unique keys from both df1 and df2 are included. NaNs fill in where there's no match.")
# print("\n" + "-"*30 + "\n")
